In [2]:
# Modelo de datos y contratos (interfaces)
# PriceRecord, SourceConnector, CurrencyConverter

from abc import ABC, abstractmethod
from dataclasses import dataclass, asdict
from datetime import datetime, date as date_type
from typing import List

@dataclass (frozen=True)
class PriceRecord: 
    """
    Esquema normalizado único, independiente de la fuente de origen.
    Toda la heterogeneidad (XML/JSON, PLN/EUR, PT15M/PT60M) se resuelve ANTES de llegar a este punto.
    """
    country: str
    datetime_utc: datetime
    price_eur: float
    granularity_minutes: int
    source: str
    ingestion_ts: datetime

    def to_dict(self) -> dict:
        #para pasar el dataclass como dict plano plano a un Dataframe 
        return asdict(self)

class SourceConnector(ABC):
    """
    Contrato (interfaz) que debe cumplir cualquier fuente de precios.
    El orquestador solo conoce esta interfaz, nunca las implementaciones concretas 
    """
    @abstractmethod
    def fetch(self, target_date: date_type) -> List[dict]:
        # Devuelve una lista de dicts crudos, normalizados a las claves: datetime_utc (datetime), price (float), 
        # currency (str), granularity_minutes (int). La conversión final a PriceRecord la hace el orquestador, 
        # no el conector
        raise NotImplementedError

class CurrencyConverter(ABC):
    @abstractmethod
    def to_eur(self, amount: float, currency: str, on_date: date_type) -> float:
        # Dado un importe, su moneda y la fecha del precio, devuelve el equivalente en EUR
        raise NotImplementedError

StatementMeta(, 4c0c52ca-38e6-4ed3-8c7e-7aa68cd35477, 4, Finished, Available, Finished, False)

In [ ]:
# Configuración de credenciales y parámetros por país
# ENTSOE_TOKEN, COUNTRY_CONFIG
import os
ENTSOE_TOKEN = os.environ.get('ENTSOE_TOKEN')  # token de ENTSO-E (mover a Fabric Key Vault en producción)

COUNTRY_CONFIG = {
    "ES": {"source": "entsoe", "domain": "10YES-REE------0", "currency": "EUR", "granularity_minutes": 15},
    "RO": {"source": "entsoe", "domain": "10YRO-TEL------P", "currency": "EUR", "granularity_minutes": 15},
    "DE": {"source": "smard", "region": "DE", "filter": "4169", "currency": "EUR", "granularity_minutes": 60},
    "PL": {"source": "pse", "currency": "PLN", "granularity_minutes": 15},
}

StatementMeta(, 4c0c52ca-38e6-4ed3-8c7e-7aa68cd35477, 5, Finished, Available, Finished, False)

In [4]:
# Conversor de Divisas (PLN -> EUR)
# NbpCurrencyConverter

import requests
from functools import lru_cache
from datetime import timedelta

class NbpCurrencyConverter(CurrencyConverter):
    """
    Usa el tipo de cambio oficial diario del Narodowy Bank Polski (NBP),
    fuente oficial y gratuita del banco central de Polonia.
    Se cachea por (currency, date) para no repetir llamadas HTTP
    si varios registros del mismo día necesitan conversión.
    """
    def to_eur(self, amount: float, currency: str, on_date: date_type) -> float:
        if currency == "EUR":
            return amount
        rate = self._get_rate(currency, on_date)
        return round(amount / rate, 4)

    @lru_cache(maxsize=256)
    def _get_rate(self, currency: str, on_date: date_type) -> float:
        date_str = on_date.strftime("%Y-%m-%d")                         # sin espacios alrededor de los guiones
        url = f"https://api.nbp.pl/api/exchangerates/rates/a/eur/{date_str}"
        resp = requests.get(url, timeout=15)
        if resp.status_code == 404:
            return self._get_rate(currency, on_date - timedelta(days=1))  # days=1, no days-1
        resp.raise_for_status()
        return resp.json()["rates"][0]["mid"]

StatementMeta(, 4c0c52ca-38e6-4ed3-8c7e-7aa68cd35477, 6, Finished, Available, Finished, False)

In [5]:
# Data Pipeline 
# valor por defecto para pruebas manuales; el pipeline lo sobrescribirá Esta será la única variable que el 
# Data Pipeline necesita tocar desde fuera.
run_date_str = "2026-09-01" 

StatementMeta(, 4c0c52ca-38e6-4ed3-8c7e-7aa68cd35477, 7, Finished, Available, Finished, False)

In [6]:
# Conector ENTSO-E España y Rumania
# EntsoeConnector

import xml.etree.ElementTree as ET

class EntsoeConnector(SourceConnector):
    """Conector para ENTSO-E. Sirve tanto para España como Rumanía."""

    BASE_URL = "https://web-api.tp.entsoe.eu/api"
    NS = {"ns": "urn:iec62325.351:tc57wg16:451-3:publicationdocument:7:3"}

    def __init__(self, domain: str, token: str, session: requests.Session = None):
        self._domain = domain
        self._token = token
        self._session = session or requests.Session()

    def fetch(self, target_date: date_type) -> List[dict]:
        params = {
            "documentType": "A44",
            "in_Domain": self._domain,
            "out_Domain": self._domain,
            "periodStart": target_date.strftime("%Y%m%d") + "0000",
            "periodEnd": (target_date + timedelta(days=1)).strftime("%Y%m%d") + "0000",
            "securityToken": self._token,
        }
        resp = self._session.get(self.BASE_URL, params=params, timeout=30)
        resp.raise_for_status()
        all_records = self._parse(resp.content)

        # ENTSO-E redondea la ventana solicitada a días de mercado completos según
        # la zona horaria local del país, por lo que puede devolver más de un día
        # si nuestra ventana UTC cruza esa frontera. Filtramos para quedarnos
        # únicamente con el día UTC exacto que pedimos.
        day_start = datetime(target_date.year, target_date.month, target_date.day)
        day_end = day_start + timedelta(days=1)
        return [r for r in all_records if day_start <= r["datetime_utc"] < day_end]

    def _parse(self, xml_data) -> List[dict]:
        root = ET.fromstring(xml_data)

        # ENTSO-E puede republicar revisiones del mismo día (correcciones de mercado).
        # Nos quedamos con el valor más reciente publicado para cada timestamp.
        records_by_ts = {}

        for period in root.findall(".//ns:Period", self.NS):
            start = datetime.strptime(
                period.find("ns:timeInterval/ns:start", self.NS).text,
                "%Y-%m-%dT%H:%MZ"
            )
            minutes = int(
                period.find("ns:resolution", self.NS).text
                .replace("PT", "").replace("M", "")
            )
            for point in period.findall("ns:Point", self.NS):
                position = int(point.find("ns:position", self.NS).text)
                price = float(point.find("ns:price.amount", self.NS).text)
                ts = start + timedelta(minutes=minutes * (position - 1))

                records_by_ts[ts] = {
                    "datetime_utc": ts,
                    "price": price,
                    "currency": "EUR",
                    "granularity_minutes": minutes,
                }

        return sorted(records_by_ts.values(), key=lambda r: r["datetime_utc"])

StatementMeta(, 4c0c52ca-38e6-4ed3-8c7e-7aa68cd35477, 8, Finished, Available, Finished, False)

In [7]:
# Conector SMARD Alemania
# SmardConnector

class SmardConnector(SourceConnector):
    """Conector para SMARD (Alemania). Requiere 2 llamadas: índice + bloque."""

    def __init__(self, filter_id: str, region: str, session: requests.Session = None):
        self._filter_id = filter_id
        self._region = region
        self._session = session or requests.Session()

    def fetch(self, target_date: date_type) -> List[dict]:
        block_ts = self._find_block(target_date)
        url = (f"https://www.smard.de/app/chart_data/{self._filter_id}/{self._region}/"
               f"{self._filter_id}_{self._region}_hour_{block_ts}.json")
        resp = self._session.get(url, timeout=30)
        resp.raise_for_status()
        return self._parse(resp.json()["series"], target_date)

    def _find_block(self, target_date: date_type) -> int:
        idx_url = f"https://www.smard.de/app/chart_data/{self._filter_id}/{self._region}/index_hour.json"
        resp = self._session.get(idx_url, timeout=30)
        resp.raise_for_status()
        target_ms = int(datetime(target_date.year, target_date.month, target_date.day).timestamp() * 1000)
        candidates = [t for t in resp.json()["timestamps"] if t <= target_ms]
        if not candidates:
            raise ValueError(f"SMARD: no hay bloque disponible para {target_date}")
        return max(candidates)

    def _parse(self, series: list, target_date: date_type) -> List[dict]:
        records = []
        target_day = target_date.date() if isinstance(target_date, datetime) else target_date  # normaliza a date, venga como venga
        for ts_ms, price in series:
            if price is None:
                continue
            ts = datetime.utcfromtimestamp(ts_ms / 1000)
            if ts.date() == target_day:                # ahora comparamos date con date
                records.append({
                    "datetime_utc": ts,
                    "price": price,
                    "currency": "EUR",
                    "granularity_minutes": 60,
                })
        return records

StatementMeta(, 4c0c52ca-38e6-4ed3-8c7e-7aa68cd35477, 9, Finished, Available, Finished, False)

In [8]:
# conector PSE Polonia
# PseConnector

class PseConnector(SourceConnector):
    """Conector para PSE (Polonia). JSON directo, ya en UTC."""

    def __init__(self, session: requests.Session = None):
        self._session = session or requests.Session()

    def fetch(self, target_date: date_type) -> List[dict]:
        url = "https://api.raporty.pse.pl/api/rce-pln"
        params = {"$filter": f"business_date eq '{target_date.strftime('%Y-%m-%d')}'"}
        resp = self._session.get(url, params=params, timeout=30)
        resp.raise_for_status()
        return self._parse(resp.json()["value"])

    def _parse(self, rows: list) -> List[dict]:
        return [
            {
                # PSE etiqueta cada franja con el timestamp de FIN de intervalo,
                # a diferencia de ENTSO-E y SMARD que usan inicio de intervalo.
                # Restamos 15 min para homogeneizar la convención entre las 4 fuentes
                # y así poder comparar precios de forma correcta en la Fase 2.
                "datetime_utc": datetime.strptime(row["dtime_utc"], "%Y-%m-%d %H:%M:%S") - timedelta(minutes=15),
                "price": row["rce_pln"],
                "currency": "PLN",
                "granularity_minutes": 15,
            }
            for row in rows
        ]



StatementMeta(, 4c0c52ca-38e6-4ed3-8c7e-7aa68cd35477, 10, Finished, Available, Finished, False)

In [9]:
# Clase conector 
# ConnectorFactory

class ConnectorFactory:
    """
    Traduce una entrada de COUNTRY_CONFIG en la instancia de conector correcta.
    Esta es la ÚNICA pieza del sistema que conoce la relación entre
    "source": "entsoe"/"smard"/"pse" y la clase concreta que la implementa.
    Añadir un país nuevo con una fuente YA soportada no requiere tocar esta clase
    (solo añadir la fila en COUNTRY_CONFIG). Añadir una fuente NUEVA sí requiere
    una línea nueva aquí + la clase conectora correspondiente — es el único punto
    de acoplamiento entre "config" y "código", y está deliberadamente centralizado.
    """

    @staticmethod                                           
    def create(country_code: str, config: dict, token: str = None) -> SourceConnector:
        source = config["source"]                              

        if source == "entsoe":
            return EntsoeConnector(domain=config["domain"], token=token)   
        elif source == "smard":
            return SmardConnector(filter_id=config["filter"], region=config["region"])  # DE
        elif source == "pse":
            return PseConnector()                              

        else:
            # si algún día se añade un país con un "source" no contemplado,
            # fallamos de forma explícita en vez de silenciosamente devolver None
            raise ValueError(f"Fuente desconocida '{source}' para el país {country_code}")

StatementMeta(, 4c0c52ca-38e6-4ed3-8c7e-7aa68cd35477, 11, Finished, Available, Finished, False)

In [10]:
# Orquestador que enlaza todo fetch - conversión - escritura
# IngestionOrchestrator

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, FloatType, IntegerType
from datetime import timedelta


class IngestionOrchestrator:
    """
    Coordina el pipeline completo para un país:
    1. Pide los datos crudos al conector correspondiente (SourceConnector)
    2. Convierte cada precio a EUR (CurrencyConverter)
    3. Construye los PriceRecord normalizados
    4. Escribe en la tabla Delta del país mediante MERGE (upsert)
    """

    SCHEMA = StructType([
        StructField("country", StringType(), nullable=False),
        StructField("datetime_utc", TimestampType(), nullable=False),
        StructField("price_eur", FloatType(), nullable=False),
        StructField("granularity_minutes", IntegerType(), nullable=False),
        StructField("source", StringType(), nullable=False),
        StructField("ingestion_ts", TimestampType(), nullable=False),
    ])

    def __init__(self, spark: SparkSession, currency_converter: CurrencyConverter, entsoe_token: str):
        self._spark = spark
        self._currency_converter = currency_converter
        self._entsoe_token = entsoe_token

    def run_for_country(self, country_code: str, config: dict, target_date: date_type) -> int:
        """Ejecuta el pipeline completo para UN país y UNA fecha concreta."""
        connector = ConnectorFactory.create(country_code, config, token=self._entsoe_token)
        raw_records = connector.fetch(target_date)

        if not raw_records:
            print(f"[WARN] {country_code}: 0 registros para {target_date}")
            return 0

        now = datetime.utcnow()
        price_records = [
            PriceRecord(
                country=country_code,
                datetime_utc=r["datetime_utc"],
                price_eur=self._currency_converter.to_eur(r["price"], r["currency"], target_date),
                granularity_minutes=r["granularity_minutes"],
                source=config["source"],
                ingestion_ts=now,
            )
            for r in raw_records
        ]

        self._write_merge(country_code, price_records)
        return len(price_records)

    def run_backfill(self, country_code: str, config: dict, start_date: date_type, end_date: date_type) -> dict:
        """
        Ejecuta run_for_country() para cada día del rango [start_date, end_date] (ambos inclusive).
        Reutiliza toda la lógica de fetch/conversión/merge sin duplicarla: el backfill
        es literalmente "un bucle sobre días" alrededor de la ingesta diaria ya validada.

        Devuelve un resumen {fecha: n_registros} para poder inspeccionar qué días
        fallaron o vinieron vacíos, sin que un solo día roto tumbe todo el rango.
        """
        results = {}                                              
        current_date = start_date                                  

        while current_date <= end_date:                            
            date_key = current_date.strftime("%Y-%m-%d")            
            try:
                n = self.run_for_country(country_code, config, current_date)  
                results[date_key] = n
                print(f"  [{country_code}] {date_key}: {n} registros")
            except Exception as e:
                # un día que falla (ej. API caída momentáneamente) no debe detener el backfill completo de los demás días del rango
                results[date_key] = f"ERROR: {e}"
                print(f"  [{country_code}] {date_key}: ❌ {e}")

            current_date = current_date + timedelta(days=1)         

        return results

    def _write_merge(self, country_code: str, records: list):
        table_name = f"{country_code.lower()}_day_ahead"
        rows = [r.to_dict() for r in records]
        df = self._spark.createDataFrame(rows, schema=self.SCHEMA)

        if not self._table_exists(table_name):
            df.write.format("delta").mode("overwrite").saveAsTable(table_name)
            print(f"[INFO] {country_code}: tabla '{table_name}' creada con {df.count()} registros")
            return

        df.createOrReplaceTempView("staging_updates")
        self._spark.sql(f"""
            MERGE INTO {table_name} AS target
            USING staging_updates AS source
            ON target.datetime_utc = source.datetime_utc
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)
        print(f"[INFO] {country_code}: MERGE completado, {len(rows)} registros procesados sobre '{table_name}'")

    def _table_exists(self, table_name: str) -> bool:
        return self._spark.catalog.tableExists(table_name)

StatementMeta(, 4c0c52ca-38e6-4ed3-8c7e-7aa68cd35477, 12, Finished, Available, Finished, False)

In [11]:
# Pipeline de ejecución de todos los paises

converter = NbpCurrencyConverter()                                   
orchestrator = IngestionOrchestrator(spark=spark, currency_converter=converter, entsoe_token=ENTSOE_TOKEN)
target_date = datetime.strptime(run_date_str, "%Y-%m-%d")                                     

for country_code, config in COUNTRY_CONFIG.items():                    
    try:
        n = orchestrator.run_for_country(country_code, config, target_date)
        print(f"✅ {country_code}: {n} registros ingeridos")
    except Exception as e:
        # capturamos el error de un país sin tumbar el pipeline completo de los demás;
        # esto es clave para robustez: un fallo en PSE no debe impedir que ES/RO/DE se ingieran
        print(f"❌ {country_code}: ERROR - {e}")

StatementMeta(, 4c0c52ca-38e6-4ed3-8c7e-7aa68cd35477, 13, Finished, Available, Finished, False)

[INFO] ES: MERGE completado, 96 registros procesados sobre 'es_day_ahead'
✅ ES: 96 registros ingeridos
[INFO] RO: MERGE completado, 96 registros procesados sobre 'ro_day_ahead'
✅ RO: 96 registros ingeridos
[INFO] DE: MERGE completado, 24 registros procesados sobre 'de_day_ahead'
✅ DE: 24 registros ingeridos
[INFO] PL: MERGE completado, 96 registros procesados sobre 'pl_day_ahead'
✅ PL: 96 registros ingeridos


In [12]:
# Exportamos cada tabla Delta a un único archivo CSV dentro de Files/exports/
# del Lakehouse. Usamos coalesce(1) para forzar un solo archivo de salida
# (por defecto Spark particiona en varios archivos), facilitando la descarga.
export_path = "Files/exports"

for country_code in COUNTRY_CONFIG:
    table_name = f"{country_code.lower()}_day_ahead"
    df = spark.table(table_name).orderBy("datetime_utc")
    
    (df.coalesce(1)
       .write
       .mode("overwrite")
       .option("header", "true")
       .csv(f"{export_path}/{table_name}_temp"))
    
    print(f"[INFO] {table_name} exportado a {export_path}/{table_name}_temp")

StatementMeta(, 4c0c52ca-38e6-4ed3-8c7e-7aa68cd35477, 14, Finished, Available, Finished, False)

[INFO] es_day_ahead exportado a Files/exports/es_day_ahead_temp
[INFO] ro_day_ahead exportado a Files/exports/ro_day_ahead_temp
[INFO] de_day_ahead exportado a Files/exports/de_day_ahead_temp
[INFO] pl_day_ahead exportado a Files/exports/pl_day_ahead_temp


In [13]:
import os

for country_code in COUNTRY_CONFIG:
    table_name = f"{country_code.lower()}_day_ahead"
    temp_dir = f"/lakehouse/default/{export_path}/{table_name}_temp"
    final_path = f"/lakehouse/default/{export_path}/{table_name}.csv"
    
    # busca el archivo part-*.csv dentro de la carpeta temporal que generó Spark
    csv_file = [f for f in os.listdir(temp_dir) if f.endswith(".csv")][0]
    os.rename(os.path.join(temp_dir, csv_file), final_path)
    
    print(f"[INFO] Renombrado a {final_path}")

StatementMeta(, 4c0c52ca-38e6-4ed3-8c7e-7aa68cd35477, 15, Finished, Available, Finished, False)

[INFO] Renombrado a /lakehouse/default/Files/exports/es_day_ahead.csv
[INFO] Renombrado a /lakehouse/default/Files/exports/ro_day_ahead.csv
[INFO] Renombrado a /lakehouse/default/Files/exports/de_day_ahead.csv
[INFO] Renombrado a /lakehouse/default/Files/exports/pl_day_ahead.csv


In [14]:
spark.sql("SELECT MIN(datetime_utc), MAX(datetime_utc) FROM es_day_ahead").show()

StatementMeta(, 4c0c52ca-38e6-4ed3-8c7e-7aa68cd35477, 16, Finished, Available, Finished, False)

+-------------------+-------------------+
|  min(datetime_utc)|  max(datetime_utc)|
+-------------------+-------------------+
|2026-08-25 00:00:00|2026-09-06 21:30:00|
+-------------------+-------------------+



In [15]:
resp = requests.get(
    EntsoeConnector.BASE_URL,
    params={
        "documentType": "A44",
        "in_Domain": "10YES-REE------0",
        "out_Domain": "10YES-REE------0",
        "periodStart": "202608260000",
        "periodEnd": "202608270000",
        "securityToken": ENTSOE_TOKEN,
    },
    timeout=30,
)

root = ET.fromstring(resp.content)
for ts in root.findall(".//ns:TimeSeries", EntsoeConnector.NS):
    for period in ts.findall(".//ns:Period", EntsoeConnector.NS):
        start = period.find("ns:timeInterval/ns:start", EntsoeConnector.NS).text
        points = period.findall("ns:Point", EntsoeConnector.NS)
        positions = sorted(int(p.find("ns:position", EntsoeConnector.NS).text) for p in points)
        print(f"Period start={start}, {len(points)} puntos, posiciones: {positions}")

StatementMeta(, 4c0c52ca-38e6-4ed3-8c7e-7aa68cd35477, 17, Finished, Available, Finished, False)

Period start=2026-08-25T22:00Z, 74 puntos, posiciones: [1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 14, 15, 16, 17, 18, 19, 21, 22, 23, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 50, 51, 53, 67, 69, 70, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 93, 94, 95, 96]
Period start=2026-08-25T22:00Z, 86 puntos, posiciones: [1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 58, 60, 61, 64, 65, 66, 67, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96]
Period start=2026-08-25T22:00Z, 89 puntos, posiciones: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 61, 6